# Análisis Exploratorio de Datos (EDA)

- **Última actualización:** 21/05/2026

## Recursos útiles
+ [Medium - A Data Scientist’s Essential Guide to Exploratory Data Analysis](https://medium.com/data-science/a-data-scientists-essential-guide-to-exploratory-data-analysis-25637eee0cf6)
+ [Medium - Mastering Exploratory Data Analysis (EDA): Everything You Need To Know](https://medium.com/data-and-beyond/mastering-exploratory-data-analysis-eda-everything-you-need-to-know-7e3b48d63a95)
+ [Towards Data Science - An Extensive Step by Step Guide to Exploratory Data Analysis](https://towardsdatascience.com/an-extensive-guide-to-exploratory-data-analysis-ddd99a03199e/)

## Objetivo 

Este notebook tiene como objetivo ilustrar paso a paso un Análisis Exploratorio de Datos (EDA) utilizando un dataset real sobre reservas de hoteles hechas a lo largo del tiempo, incluyendo detalles sobre los clientes, el comportamiento de reserva y la probabilidad de cancelación. 

El objetivo será predecir si una reserva será cancelada (is_canceled = 1) o no (is_canceled = 0 ).

## Descripción de los Datos

Este conjunto de datos contiene información sobre reservas de hoteles, incluyendo la fecha de la reserva, la duración de la estancia, el número de huéspedes y detalles sobre cancelaciones. 

<details>
<summary><b>🔍 Click aquí para ver el Diccionario de Variables</b></summary>


| Nombre Variable                  | Descripción                                              |
| -------------------------------- | -------------------------------------------------------- |
|  hotel                           | Tipo de hotel: City Hotel o Resort Hotel                 |
|  is_canceled                     | Variable objetivo: 1 si fue cancelado, 0 si no           |
|  lead_time                       | Días entre la reserva y la fecha de llegada              |
|  arrival_date_year               | Año de llegada                                           |
|  arrival_date_month              | Mes de llegada                                           |
|  arrival_date_week_number        | Número de la semana del año                              |
|  arrival_date_day_of_month       | Día del mes de llegada                                   |
|  stays_in_weekend_nights         | Noches de fin de semana reservadas                       |
|  stays_in_week_nights            | Noches entre semana reservadas                           |
|  adults                          | Número de adultos                                        |
|  children                        | Número de niños                                          |
|  babies                          | Número de bebés                                          |
|  meal                            | Tipo de comida reservada                                 |
|  country                         | País de origen del cliente                               |
|  market_segment                  | Canal de marketing (online, offline, grupos...)          |
|  distribution_channel            | Canal de distribución (directo, TA/TO...)                |
|  is_repeated_guest               | 1 si el cliente ha estado anteriormente                  |
|  previous_cancellations          | Nº de cancelaciones anteriores                           |
|  previous_bookings_not_canceled  | Nº de reservas previas no canceladas                     |
|  reserved_room_type              | Tipo de habitación reservada                             |
|  assigned_room_type              | Tipo de habitación asignada                              |
|  booking_changes                 | Nº de cambios en la reserva                              |
|  deposit_type                    | Tipo de depósito: No Deposit, Refundable, etc.           |
|  agent                           | ID del agente (puede ser nulo)                           |
|  company                         | ID de la empresa (puede ser nulo)                        |
|  days_in_waiting_list            | Días en lista de espera                                  |
|  customer_type                   | Tipo de cliente: Transient, Group, etc.                  |
|  adr                             | Average Daily Rate (precio promedio por noche)           |
|  required_car_parking_spaces     | Plazas de parking solicitadas                            |
|  total_of_special_requests       | Nº de peticiones especiales                              |
|  reservation_status              | Estado final de la reserva: Check-Out, Canceled, No-Show |
|  reservation_status_date         | Fecha en que se actualizó el estado                      |

</details>

## Configuración del notebook 

In [ ]:
## Carga de librerias 
import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
import seaborn as sns

from plotly.subplots import make_subplots

from ydata_profiling import ProfileReport

## Definicion de constantes
# Paths
PATH_DIRECTORIO_DATOS = "../data/raw"

PATH_DATASET_HOTELS = f"{PATH_DIRECTORIO_DATOS}/dataset_practica_final.csv"

# Preprocesados
DICT_MAP_TARGET = {0: 'No cancelable', 1: 'Cancelable'}

## Definimos paleta de colores
# Mostramos las paletas de colores que tiene Plotly
px.colors.qualitative.swatches()

# Definimos una paleta de color según los valores del target
LIST_PALETA_COLOR = px.colors.qualitative.T10

# Asignamos los índices de dicha paleta a los valores del target
# is_canceled = 0 -> color verde (porque es buen indicador)
# is_canceled = 1 -> color rojo (porque es mal indicador)
DICT_PALETA_COLOR_TARGET = {'No cancelable': LIST_PALETA_COLOR[4], 'Cancelable': LIST_PALETA_COLOR[2]}

In [ ]:
PATH_DATASET_HOTELS

## 📊 Carga y exploración inicial de los datos

In [ ]:
# Cargar el dataset
df = pd.read_csv(PATH_DATASET_HOTELS)
df.head()

In [ ]:
# Se muestran los tipos de variables del dataset
df.info()

In [ ]:
# Copiamos el dataframe para no perder el original
df_preprocessed = df.copy()

In [ ]:
print(f"Filas con valores duplicados: {df_preprocessed.duplicated().sum()}")
df_preprocessed.shape

### 🚫 Valores nulos

In [ ]:
print("Valores vacíos por cada columna:")
print(df_preprocessed.isna().sum()[df_preprocessed.isna().sum() > 0])

In [ ]:
# Vamos columna a columna para analizar cada una de ellas. Empezamos por "children"
# Analizamos la distribución de la variable "children", para ver si tiene sentido imputar los valores nulos a un valor concreto
sns.histplot(data=df_preprocessed, x="children", discrete=True, color="skyblue", shrink=0.8)

In [ ]:
# Procesamos y colocamos 0 niños a las familias que no han indicado el número de niños. De esta forma, el modelo entenderá que no tienen niños, 
# en lugar de interpretar el valor nulo como un valor desconocido.
# Por que 0? Porque es el valor que tiene más sentido en este contexto. Si una familia no ha indicado el número de niños, lo más probable es que no tengan niños, 
# por lo que asignarles un valor de 0 es una suposición razonable. 
df_preprocessed['children'] = df_preprocessed['children'].fillna(0)

In [ ]:
# Vamos ahora a por las variable "company". Se imputan los valores nulos como 'No Company' 
# ya que la ausencia de ID indica transacciones/clientes directos.
df_preprocessed["company"] = df_preprocessed["company"].fillna("no company")

In [ ]:
# HAcemos lo mismo con "agent", imputando los nulos como 'No Agent'
df_preprocessed["agent"] = df_preprocessed["agent"].fillna("no agent")

In [ ]:
# A diferencia de 'company' y 'agent, el vacío en 'country' representa una omisión en el registro 
# del dato geográfico, no una condición de "no aplica". 
pais_mas_frecuente = df_preprocessed["country"].mode()[0]
print(f"El país más frecuente es: {pais_mas_frecuente} con una frecuencia de {df_preprocessed['country'].isna().mean() * 100} % de registros.")

# Como el porcentaje de nulos es relativamente bajo, 
# imputar el país más frecuente es una estrategia razonable para mantener la integridad del dataset sin introducir sesgos significativos.
df_preprocessed["country"] = df_preprocessed["country"].fillna(pais_mas_frecuente)

In [ ]:
# Comprobamos que haya quedado todo correcto
print("Valores vacíos por cada columna:")
print(df_preprocessed.isna().sum()[df_preprocessed.isna().sum() > 0])

### Variables categoricas y numericas

In [ ]:
# Quiero utilizar como numerica la columna arrival_date_month,asi que vamos a procesarla
# El objetivo es que el modelo entienda la estacionalidad
print(df_preprocessed['arrival_date_month'].unique())

In [ ]:
meses_map = {
    'January': 1, 'February': 2, 'March': 3, 'April': 4,
    'May': 5, 'June': 6, 'July': 7, 'August': 8,
    'September': 9, 'October': 10, 'November': 11, 'December': 12}
df_preprocessed['arrival_date_month'] = df_preprocessed['arrival_date_month'].str.strip()
df_preprocessed['arrival_date_month'] = df_preprocessed['arrival_date_month'].map(meses_map) 

In [ ]:
# Tambien nos damos cuenta de que hay dos columnas que no deberiamos usar para la 
# prediccion, puesto que aporta info una vez cancelada. Por tanto la eliminamos del df y 
# no las meteremos en los listados de columnas 
# Ambas variables se generan como consecuencia de la cancelación, no como predictores de ella.
df_preprocessed.drop(columns=['reservation_status_date','reservation_status'], inplace=True)

In [ ]:
# Con un poco mas de info del dataframe identificamos columnas categoricas y numericas. Asi como la variable target
LIST_CAT_COLS = ['hotel', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'reserved_room_type', 'assigned_room_type','deposit_type', 'agent', 'company', 'customer_type']
LIST_NUM_COLS = ['lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children','babies', 'previous_cancellations', 'previous_bookings_not_canceled', 'booking_changes', 'days_in_waiting_list', 'adr', 'required_car_parking_spaces','total_of_special_requests']
VARIABLE_TARGET = 'is_canceled'
assert len(LIST_CAT_COLS + LIST_NUM_COLS) + 1 == df_preprocessed.shape[1], "Faltan columnas"

In [ ]:
# Se convierten las variables que deberían ser categóricas a formato str
df_preprocessed[LIST_CAT_COLS] = df_preprocessed[LIST_CAT_COLS].astype(str)

### 📈 Estadísticos descriptivos

In [ ]:
# Estadísticos de las variables numéricas
df_preprocessed.describe().transpose()

In [ ]:
# Estadísticos de las variables categóricas
df_preprocessed.describe(include='object').transpose()

### Convertimos los valores numéricos de la columna target (is_canceled) a tipo objeto
Usamos el diccionario previamente almacenado en una constante:

In [ ]:
# Se convierte a tipo entero de pandas (soporta NaN)
df_preprocessed[VARIABLE_TARGET] = df_preprocessed[VARIABLE_TARGET].astype("Int64")
df_preprocessed[VARIABLE_TARGET] = df_preprocessed[VARIABLE_TARGET].map(DICT_MAP_TARGET)

## 🔎 Análisis de valores

### 🔎 Análisis de valores duplicados

In [ ]:
print(f"Filas con valores duplicados: {df_preprocessed.duplicated().sum()}")

Se han detectado **31,994 filas duplicadas** (aprox. el 27% del dataset). 

* **Interpretación de Negocio:** El 27% del dataset nos parece muchisimo como para borrarlo muy a la ligera. Según parece en el sector hotelero se suele proceder así para reservas de grupos (reservas en bloque -grupos, turoperadores o familias-) que reservan varias habitaciones idénticas para las mismas fechas.
* **Decisión de Diseño:** Eliminar el 27% de los datos no parece viable. Sin embargo, mantenerlas tal cual generaría *Data Leakage* (fuga de datos) y sobreajuste en el modelo. Por la separacion train/test.
* **Solución:** Agrupamos el dataset por todas sus columnas para colapsar los duplicados, pero guardamos el volumen de la reserva en una nueva variable (`room_count`). De esta forma, protegemos la integridad matemática del modelo sin perder la señal de comportamiento grupal.

In [ ]:
# Gestionamos los valores duplicados
df_preprocessed = df_preprocessed.value_counts(dropna=False).reset_index(name='room_count')


# Ahora si borramos duplicados
df_preprocessed = df_preprocessed.drop_duplicates()
print(f"Antes teniamos {df.shape[0]} filas y ahora {df_preprocessed.shape[0]}")

### 🔎 Análisis de valores en los precios (adr) raros

In [ ]:
# Ver cuántos registros negativos hay afectados
print(f"adr negativo: {(df_preprocessed['adr'] < 0).sum()}")

# Para los outliers por "arriba", usamos con percentil razonable el 99
percentil_99 = df_preprocessed['adr'].quantile(0.99)
print(f"Percentil 99 de adr: {percentil_99}")
# Contamos el numero que supera el P99
print(f"adr > {percentil_99}: {(df_preprocessed['adr'] > 261).sum()}")

In [ ]:
# Eliminamos los negativos, que son claramente erróneos
df_preprocessed = df_preprocessed[df_preprocessed['adr'] >= 0]
# Borramos los registros que superan el percentil 99
df_preprocessed = df_preprocessed[df_preprocessed['adr'] <= percentil_99]

Conclusion: Se eliminan registros con adr negativo por ser errores de registro y se aplica tambien a aquellos registros que superan el percentil 99.

### 🔎 Análisis de valores posibles en los country

In [ ]:
# Vamos los 11 paises mas frecuentes, para ver si es necesario agrupar los paises menos frecuentes en una categoria "OTHER"
top11_count = df_preprocessed['country'].value_counts().head(11).sum()
total = df_preprocessed['country'].shape[0]
print(f"Top 11 países: {top11_count/total*100:.1f}% de las reservas")
# El listado de los 11 países más frecuentes
top_countries = df_preprocessed['country'].value_counts().head(11).index.tolist()
print(f"Países: {top_countries}")

In [ ]:
df_preprocessed['country'] = df_preprocessed['country'].where(
    df_preprocessed['country'].isin(top_countries), other='OTHER')

# Verificamos
print(df_preprocessed['country'].value_counts())

### 🔎 Análisis de valores en agent y company
En estas columnas hay valores de IDs de agentes y empresas. No parece que vaya aportar valor. 
Se puede cambiar por has_agent y has_company, columnas booleanas

In [ ]:
print(f"Sin agente: {df_preprocessed['agent'].value_counts(normalize=True)['no agent']*100:.1f}%")
print(f"Sin empresa: {df_preprocessed['company'].value_counts(normalize=True)['no company']*100:.1f}%")

In [ ]:
# Creamos nuevas variables binarias para indicar si el cliente tiene agente o empresa, ya que la ausencia de estos puede ser un indicador relevante para la cancelación.
df_preprocessed['has_agent'] = (df_preprocessed['agent'] != 'no agent').astype(int)
df_preprocessed['has_company'] = (df_preprocessed['company'] != 'no company').astype(int)

# Eliminamos las columnas originales de 'agent' y 'company' ya que ahora tenemos las variables binarias que indican su presencia o ausencia.
df_preprocessed.drop(columns=['agent'], inplace=True)
df_preprocessed.drop(columns=['company'], inplace=True)


### 🔎 Análisis de valores en market_segment
En estas columnas hay valores de Undefined. Vamos a ver cuanto peso tienen en la muestra para decidir si eliminamos estas filas.

In [ ]:
# Contar cuántas filas son 'Undefined' en market_segment
num_undefined = (df_preprocessed['market_segment'] == 'Undefined').sum()
pct_undefined = (num_undefined / len(df_preprocessed)) * 100

print(f"Cantidad de filas 'Undefined': {num_undefined}")
print(f"Porcentaje sobre el total: {pct_undefined:.4f}%")

In [ ]:
# Eliminamos
df_preprocessed = df_preprocessed[df_preprocessed['market_segment'] != 'Undefined'].copy()

## Análisis univariado

### Variables numéricas

In [ ]:
# # Se muestran las distribuciones de las variables numéricas
# for col in LIST_NUM_COLS:
#     fig = ff.create_distplot(
#         [df_preprocessed[col]],
#         [col],
#         show_hist=True,
#         show_curve=True,
#         show_rug=False
#     )
#     fig.update_layout(
#         title=f"Distribución variable: {col}"
#     )
#     fig.show()

### 📃 Variables categóricas

In [ ]:
# for col in LIST_CAT_COLS:
#     fig = make_subplots()

# for col in LIST_CAT_COLS:
#     fig = px.histogram(
#         df_preprocessed,
#         x=col,
#         text_auto=True,
#         title=f'Distribución de la variable categórica: {col}',
#         color_discrete_sequence=['mediumturquoise']
#     )
#     fig.show()

## 📊 4. Variable objetivo: `target`

In [ ]:
# # Crear subplots con tipos específicos
# fig = make_subplots(
# 	rows=1, cols=2, 
# 	subplot_titles=("Frecuencias absolutas", "Frecuencias relativas"), 
# 	specs=[[{"type": "bar"}, {"type": "pie"}]]
# )

# # Creamos el gráfico de barras (Absolutas)
# df_count = df[VARIABLE_TARGET].value_counts().reset_index()
# fig_count = px.bar(df_count, x=VARIABLE_TARGET, y="count")

# # 2. Creamos el gráfico de sectores (Relativas / Proporciones)
# fig_prop = px.pie(df_count, names=VARIABLE_TARGET, values="count")

# # Añadir las trazas del gráfico de barras
# for trace in fig_count.data:
# 	fig.add_trace(trace, row=1, col=1)

# # Añadir las trazas del gráfico de sectores
# for trace in fig_prop.data:
# 	fig.add_trace(trace, row=1, col=2)

# # Actualizar layout
# fig.update_layout(
# 	title_text=f"Distribución de la variable objetivo: {VARIABLE_TARGET}",
# 	showlegend=False
# )

# fig.show()

## Análisis Multivariado: El Pairplot
El **Pairplot** es una matriz de gráficos que nos permite visualizar la relación entre múltiples variables numéricas simultáneamente. Es una herramienta clave para la **Ingeniería de Características (Feature Engineering)**.

### Componentes del Pairplot:
1.  **Diagonal Principal:** Muestra la distribución univariada de cada variable (usualmente mediante un histograma o un KDE - *Kernel Density Estimate*).
2.  **Fuera de la diagonal:** Muestra **gráficos de dispersión (scatter plots)** entre pares de variables.
    * **Correlación:** Permite ver si dos variables se mueven juntas (ej. a mayor `lead_time`, ¿hay más cancelaciones?).
    * **Separabilidad:** Al usar el parámetro `hue='is_canceled'`, podemos ver visualmente qué variables discriminan mejor entre una reserva que se cancela y una que no.

> **Nota técnica:** Debido al alto número de registros (119,390) y variables (32), para el Pairplot es recomendable seleccionar un subconjunto de variables clave y una muestra aleatoria de los datos para optimizar el rendimiento computacional.

In [ ]:
# # Seleccionamos solo las variables numéricas clave para no saturar
# variables_clave = ['lead_time', 'adr', 'total_of_special_requests', 'booking_changes', 
#                    'required_car_parking_spaces','previous_cancellations','stays_in_week_nights','is_canceled']

# # Creamos el gráfico usando solo una muestra de 2000 filas
# sns.pairplot(
#     df_preprocessed[variables_clave].sample(2000, random_state=42), 
#     hue='is_canceled', 
#     diag_kind='kde', # Curvas suaves en la diagonal
#     plot_kws={'alpha': 0.5} # Puntos semitransparentes para ver la densidad
# )

## Matriz de correlación

In [ ]:
# Convertimos la variable objetivo a formato numérico para poder calcular la correlación
df_preprocessed[VARIABLE_TARGET] = df_preprocessed[VARIABLE_TARGET].map({'No cancelable': 0, 'Cancelable': 1})

# Calculamos la correlación entre las variables
corr = df_preprocessed[LIST_NUM_COLS + [VARIABLE_TARGET]].corr().round(2)
z = corr.values
x = list(corr.columns)
y = list(corr.index)

In [ ]:
# Mostramos la matriz de correlación con Seaborn y Matplotlib
plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Matriz de correlación")
plt.show()

## 🔄 Análisis bivariado: relación con la variable objetivo
El **Análisis Bivariado** consiste en estudiar la relación entre dos variables. En este proyecto, nuestro interés principal es entender cómo influye cada característica del dataset en la probabilidad de que una reserva sea cancelada.

Las variables que aparentemente tienen una correlacion mayor con is_canceled son:
lead_time, adr, required_parking_spaces

Ademas tambien podemos decir que # arrival_date_month y arrival_date_week_number tienen correlacion de 1.0 (correlación perfecta)


### Variables

In [ ]:
for col in ['lead_time', 'adr', 'required_car_parking_spaces']:
    print(f"\n--- {col} ---")
    print(df_preprocessed.groupby("is_canceled")[col].describe().round(2))

Conclusiones
**lead_time**
* Mediana canceladas: 80 días vs no canceladas: 38 días
* a mayor antelación, mayor probabilidad de cancelación

**adr**
* Mediana canceladas: 108€ vs no canceladas: 93€
* Las reservas más caras se cancelan más

**required_car_parking_spaces**
* ninguna reserva cancelada pidió parking
* Pedir parking es un indicador fortísimo de que el cliente va a aparecer

In [ ]:
# CONCLUSION: A la vista de los datos
# arrival_date_month y arrival_date_week_number tienen correlacion de 1.0 (correlación perfecta)
# Son dos formas de decir lo mismo. Nos quedamos solo con una: arrival_date_month
df_preprocessed.drop(columns=['arrival_date_week_number'], inplace=True)
LIST_NUM_COLS.remove('arrival_date_week_number')

In [ ]:
# Creamos el perfil de datos con ydata-profiling, que es una herramienta de EDA automática que genera un informe completo con estadísticas descriptivas, distribuciones, correlaciones y más.
# profile = ProfileReport(df, title="EDA Automático - Heart Disease")
# profile.to_notebook_iframe()